In [1]:
import logging
import time

import win32com.client  # Windows COM 인터페이스 사용
import xml.etree.ElementTree as ET
import pandas as pd

import os.path

# 입력 파라미터
###################################################################################################################
network_path = r"E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\Test\화성~서울(시나리오 분석)_260330.inpx"

csv_path = r"C:\Users\(주)내일이엔시 도로교통안전연구소\Desktop\프로젝트\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\1. 회의자료\25.12.23"
file_name = r"시나리오 테이블(적용).csv"

log_path = r"E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\Test\vissim_log.txt"

logging.basicConfig(
    filename=log_path,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

# 유입램프 갯수
ramp_in_num = 2

# 유출램프 갯수
ramp_out_num = 3

# 유고 발생 시각
acc_start_time = 3600
###################################################################################################################

full_path  = os.path.join(csv_path, file_name)

# 시나리오 파일 읽기
senario_df = pd.read_csv(full_path)
logging.info("시나리오 파일 로드 완료")

Vissim = win32com.client.Dispatch("Vissim.Vissim")  # Vissim 인스턴스 생성

for index, row in senario_df.iloc[3:].iterrows():

    logging.info(f"[START] scenario index={index}")

    # 교통량
    Q_main_in_vehph = row["Q_main_in_vehph"]
    Q_ramp_in_per_ramp = row["Q_ramp_in_per_ramp"]

    # 램프 유출 비율
    Q1_ramp_out_vehph = row["Q1_ramp_out_vehph"]
    Q2_ramp_out_vehph = row["Q2_ramp_out_vehph"]
    Q3_ramp_out_vehph = row["Q3_ramp_out_vehph"]

    # 본선 유입 중차량 비율
    V1_main_in_ratio = row["V1_main_in_ratio"]
    V2_main_in_ratio = row["V2_main_in_ratio"]
    V3_main_in_ratio = row["V3_main_in_ratio"]
    V4_main_in_ratio = row["V4_main_in_ratio"]
    V5_main_in_ratio = max(row["V5_main_in_ratio"], 0.001)

    # 램프 유입 중차량 비율
    V1_ramp_in_ratio = row["V1_ramp_in_ratio"]
    V2_ramp_in_ratio = row["V2_ramp_in_ratio"]
    V3_ramp_in_ratio = row["V3_ramp_in_ratio"]
    V4_ramp_in_ratio = row["V4_ramp_in_ratio"]
    V5_ramp_in_ratio = max(row["V5_ramp_in_ratio"], 0.001)

    # 종단경사
    grade_percent = row["grade_percent"] * 0.01

    # 유고 지속시간
    incident_duration_min = row["incident_duration_min"]

    # 유고 차로수
    lane_closure_count = int(row["lane_closure_count"])

    # 랜덤시드
    random_seed = int(row["random_seed"])


    logging.info(
            f"입력값 | main={Q_main_in_vehph}, ramp={Q_ramp_in_per_ramp}, "
            f"duration={incident_duration_min}, lane={lane_closure_count}, seed={random_seed}"
        )


    # XML 로드
    tree = ET.parse(network_path)
    root = tree.getroot()


    # 본선, 유입램프 교통량
    for vehicle_input in root.findall(".//vehicleInput"):
        input_name = vehicle_input.get("name")
        vehicleInput = vehicle_input.find("timeIntVehVols")

        if vehicleInput is None:
            continue
        for vol in vehicleInput.findall("timeIntervalVehVolume"):
            if input_name == "main":
                vol.set("volume", str(Q_main_in_vehph))
            elif input_name in ["ramp1-1", "ramp1-2"]:
                vol.set("volume", str(Q_ramp_in_per_ramp))

    # 본선/ 램프 유입 중차량 비율
    for vehicle_comp in root.findall(".//vehicleComposition"):
        if vehicle_comp.get("name") == "main":
            relflow_100 = V1_main_in_ratio
            relflow_300 = V2_main_in_ratio
            relflow_630 = V3_main_in_ratio
            relflow_640 = V4_main_in_ratio
            relflow_650 = V5_main_in_ratio

        elif vehicle_comp.get("name") == "ramp":
            relflow_100 = V1_ramp_in_ratio
            relflow_300 = V2_ramp_in_ratio
            relflow_630 = V3_ramp_in_ratio
            relflow_640 = V4_ramp_in_ratio
            relflow_650 = V5_ramp_in_ratio
        else:
            continue
        # <vehCompRelFlows> 태그 찾기
        vehCompRelFlows = vehicle_comp.find("vehCompRelFlows")
        if vehCompRelFlows is not None:
            # <vehicleCompositionRelativeFlow> 태그 찾기
            for relflow in vehCompRelFlows.findall("vehicleCompositionRelativeFlow"):
                vehType = relflow.get("vehType")
                if vehType == "100": # 승용차
                    relflow.set("relFlow", str(relflow_100))
                elif vehType == "300": # 버스
                    relflow.set("relFlow", str(relflow_300))
                elif vehType == "630": # 소형화물
                    relflow.set("relFlow", str(relflow_630))
                elif vehType == "640": # 중형화물
                    relflow.set("relFlow", str(relflow_640))
                elif vehType == "650": # 대형화물
                    relflow.set("relFlow", str(relflow_650))

        # 유고 지점 종단경사
    for link in root.findall(".//link"):
        if link.get("name") == "유고":  # 특정 no 값을 가진 <link> 찾기
            link.set("gradient", str(grade_percent))  # gradient 속성 변경

    if lane_closure_count == 1 :
        one_lane_time_from = 99999
        one_lane_time_to = 99999


    for rsa in root.findall(".//reducedSpeedArea"):
        name = rsa.get("name")

        # 1차로 폐쇄
        if lane_closure_count == 1 :
            if name == "정지1":
                rsa.set("timeFrom", str(acc_start_time))
                rsa.set("timeTo", str(int(acc_start_time + incident_duration_min * 60)))
            elif name == "정지2":
                rsa.set("timeFrom", str(99998))
                rsa.set("timeTo", str(99999))

        elif lane_closure_count == 2 :
            if name == "정지1":
                rsa.set("timeFrom", str(acc_start_time))
                rsa.set("timeTo", str(int(acc_start_time + incident_duration_min * 60)))
            elif name == "정지2":
                rsa.set("timeFrom", str(acc_start_time))
                rsa.set("timeTo", str(int(acc_start_time + incident_duration_min * 60)))
        else :
            continue

    for rds in root.findall(".//simulation"):
        rds.set("randSeed", str(random_seed))

    tree.write(network_path, encoding="utf-8", xml_declaration=True)
    logging.info("XML 설정 완료")

    # VISSIM 네트워크 로드
    Vissim.LoadNet(network_path)
    Vissim.Graphics.CurrentNetworkWindow.SetAttValue('QuickMode', 1) # 퀵 모드 활성화

    logging.info("XML 설정 완료")
    logging.info("시뮬레이션 실행 시작")

    print("VISSIM 시뮬레이션 실행 중...")
    Vissim.Simulation.RunContinuous()
    logging.info("시뮬레이션 실행 완료")
    print("VISSIM 시뮬레이션 실행 완료")

Vissim.Simulation.Stop()
Vissim.Exit()
print("VISSIM 시뮬레이션 종료")
logging.info("VISSIM 종료")
print("==============================================================================================================")




VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 실행 완료
